<!--NOTEBOOK_HEADER-->
*This notebook contains material from [PyRosetta](https://RosettaCommons.github.io/PyRosetta.notebooks);
content is available [on Github](https://github.com/RosettaCommons/PyRosetta.notebooks.git).*

<!--NAVIGATION-->
< [PyRosettaCluster Tutorial 4. Ligand params](http://nbviewer.jupyter.org/github/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.10-PyRosettaCluster-Ligand-params.ipynb) | [Contents](toc.ipynb) | [Index](index.ipynb) | [Command Reference](http://nbviewer.jupyter.org/github/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/A.00-Appendix-A.ipynb) ><p><a href="https://colab.research.google.com/github/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.11-PyRosettaCluster-Foundry.ipynb"><img align="left" src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" title="Open in Google Colaboratory"></a>

# PyRosettaCluster Tutorial 5. Foundry

PyRosettaCluster Tutorial 5 shows an example of how to execute a complete PyRosettaCluster workflow, starting from generating designs with *RFdiffusion3*, *ProteinMPNN*, *RosettaFold-3*, and *PyRosetta* in a *Pixi* environment, and subsequently showing how to export a PyRosetta initialization file from a PyRosettaCluster output decoy file, then use it to recreate the original *Pixi* environment and reproduce the decoy of interest.

*Note:* This Jupyter notebook uses parallelization and is preferrably executed within a **Google Colab** environment, but may still be run in JupyterLab or as a standalone Jupyter notebook.

*Note:* This Jupyter notebook requires the PyRosetta distributed layer which is obtained by building PyRosetta with the `--serialization` flag or installing PyRosetta from the RosettaCommons conda channel.

*Note:* use of one or more GPUs with *Foundry* cannot guarantee reproducibility of the decoy of interest due to variability
presumably caused by floating-point non-associativity and/or unordered parallel reductions. *Challenge:* experiment with different user-defined PyRosetta protocols (e.g., remove the first PyRosetta protocol containing the *RFdiffusion3* step and input a backbone of interest instead) in order to show reproducibility of the simulation on GPUs.

*Note:* This notebook automatically installs the *Pixi* environment manager if is not already installed. Within Pixi projects, this notebook installs  third-party dependencies from Conda channels and PyPI. This notebook also automatically sets `pandas` and `pyarrow` as secure packages in PyRosetta.

**References:**

1. Klima, J. C. PyRosettaCluster: a Python framework for scalable and reproducible bio-macromolecular modeling and design. ChemRxiv. 1 May 2026. https://doi.org/10.26434/chemrxiv.15002628/v1.
2. Klima, J. C. PyRosettaCluster: A Python Framework for Scalable and Reproducible Bio-macromolecular Modeling and Design: Supplementary Materials. Zenodo, 1 May 2026. https://doi.org/10.5281/zenodo.19828564.
3. Corley, N., Mathis, S., Krishna, R., Bauer, M.S., Thompson, T.R., Ahern, W., Kazman, M.W., Brent, R.I., Didi, K., Kubaney, A. and McHugh, L. Accelerating biomolecular modeling with AtomWorks and RF3. bioRxiv, 2025. https://doi.org/10.1101/2025.08.14.670328.
4. Butcher, J., Krishna, R., Mitra, R., Brent, R.I., Li, Y., Corley, N., Kim, P.T., Funk, J., Mathis, S. and Salike, S. De novo design of all-atom biomolecular interactions with rfdiffusion3. bioRxiv, 2025. https://doi.org/10.1101/2025.09.18.676967.
5. Dauparas, J., Anishchenko, I., Bennett, N., Bai, H., Ragotte, R.J., Milles, L.F., Wicky, B.I., Courbet, A., de Haas, R.J., Bethel, N. and Leung, P.J. 2022. Robust deep learning–based protein sequence design using ProteinMPNN. Science, 378(6615), pp.49-56. https://doi.org/10.1126/science.add2187.
6. Dauparas, J., Lee, G.R., Pecoraro, R., An, L., Anishchenko, I., Glasscock, C. and Baker, D., 2025. Atomic context-conditioned protein sequence design using LigandMPNN. Nature Methods, 22(4), pp.717-723. https://doi.org/10.1038/s41592-025-02626-1.

In [ ]:
import json
import os
import re
import requests
import shutil

from pathlib import Path

# Stage I: Original Simulation



In [ ]:
# @title Enter the PyRosettaCluster output directory for the original simulation
# @markdown ##### E.g., enter `/content/original_data`. If the path begins with `/content/drive/MyDrive`, you will be asked to mount Google Drive
original_output_path = "/content/original_data" # @param {type:"string"}
original_output_path = Path(original_output_path)

if not os.getenv("DEBUG"):
    if str(original_output_path).startswith("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")

    try:
        original_output_path.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        print(f"Please remove the output directory and try again: {original_output_path}")

##### Here, we use the Pixi environment manager to create a virtual environment for executing the PyRosettaCluster simulation.

##### For further details, see: https://github.com/RosettaCommons/pyrosetta-extras/tree/main/pyrosettacluster#creating-environments-for-pyrosettacluster

In [ ]:
# @title Install the Pixi environment manager

def get_pixi_version(default_version="v0.75.0"):
    """Get the latest released Pixi version if possible, otherwise fallback to a pre-defined version."""
    if os.getenv("DEBUG"):
        return default_version
    response = requests.get("https://api.github.com/repos/prefix-dev/pixi/releases/latest")
    return response.json()["tag_name"] if response.status_code == 200 else default_version

# Set the Pixi version to be used throughout the notebook
PIXI_VERSION = get_pixi_version()

if not os.getenv("DEBUG"):
    # Install Pixi
    if not shutil.which("pixi"):
        !export PIXI_VERSION={PIXI_VERSION} && curl -fsSL https://pixi.sh/install.sh | sh
        os.environ["PATH"] = f"{os.getenv('PATH')}{os.pathsep}/root/.pixi/bin"
    !pixi --version

In [ ]:
# @title Create the original Pixi project
# Warning: this cell can take ~5+ minutes for installation to complete

pixi_toml = """
[workspace]
channels = [
    "https://conda.rosettacommons.org",
    "https://conda.graylab.jhu.edu",
    "conda-forge",
]
platforms = {platform}

[dependencies]
pyrosetta = "*"
python = "*"

[pypi-dependencies]
pyrosetta-distributed = ">=0.0.5"
rc-foundry = {{ version = ">=0.1.9", extras = ["all"] }}
"""

if not os.getenv("DEBUG"):
    # Setup Pixi project
    platform = !pixi info --json | jq -r ".platform" # Auto-detect platform
    original_manifest_path = Path.cwd() / "pixi.toml"
    original_manifest_path.write_text(pixi_toml.format(platform=platform))
    original_env_file = Path.cwd() / ".env"
    original_env_file.write_text(
        "\n".join(
            [
                "PDB_MIRROR_PATH=",
                "CCD_MIRROR_PATH=",
                "LOCAL_MSA_DIRS=",
                "HBPLUS_PATH=",
                "X3DNA_PATH=",
                "DSSP_PATH=",
                "HHFILTER_PATH=",
                "MMSEQS2_PATH=",
                "COLABFOLD_LOCAL_DB_PATH_GPU=",
                "COLABFOLD_LOCAL_DB_PATH_CPU=",
                "COLABFOLD_NET_DB_PATH_GPU=",
                "COLABFOLD_NET_DB_PATH_CPU=",
                "FOUNDRY_CHECKPOINT_DIRS=",
            ]
        )
    )
    if (original_manifest_path.parent / "pixi.lock").exists():
        !pixi install --manifest-path {original_manifest_path} --frozen --no-progress
    else:
        !pixi install --manifest-path {original_manifest_path} --no-progress

In [ ]:
# @title Create a Git repository for user-defined PyRosetta protocols

# The Git repository will remain local for this tutorial,
# but for real experiments, you may wish to push the repository to GitHub
git_repo = Path.cwd() / "my_git_repo"
git_repo.mkdir(parents=True, exist_ok=False)

if not os.getenv("DEBUG"):
    !git config --global user.name "Your Name" # Optionally add your name
    !git config --global user.email "you@example.com" # Optionally add your email
    !git config --global init.defaultBranch main
    !git init {git_repo}
    !curl https://raw.githubusercontent.com/github/gitignore/main/Python.gitignore > {git_repo}/.gitignore

In [ ]:
%%writefile {git_repo}/my_utils.py
# @title Write helper functions

import hashlib
import json
import os
import pandas as pd
import pyrosetta
import pyrosetta.distributed.io as io
import time
import torch

from biotite.structure import AtomArray
from biotite.structure.io.pdb import PDBFile
from functools import wraps
from io import StringIO
from pathlib import Path
from pyrosetta.distributed.packed_pose.core import PackedPose
from typing import Any, Callable, Optional, TypeVar, cast

T = TypeVar("T", bound=Callable[..., Any])


def timeit(func: T) -> T:
    """
    Decorator that prints the runtime of a function after it finishes.

    Args:
        func: A required callable to be timed.

    Returns:
        A callable with the same function signature and return type as `func`.
    """
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        """Wrapper function that times `func` and prints its runtime."""
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        t1 = time.perf_counter()
        dt = t1 - t0
        print(f"The `{func.__name__}` function finished in {dt:.3f} seconds.")

        return result

    return cast(T, wrapper)


def pyrosetta_to_torch_seed(pyrosetta_seed: int) -> int:
    """
    Scale an input PyRosetta seed to the Torch seed proper range.
    PyRosetta seed range: [-(2 ** 31), (2 ** 31) - 1]
    Torch seed range: [0, (2 ** 32) - 1]

    Args:
        pyrosetta_seed: An `int` object representing the PyRosetta seed.

    Returns:
        An `int` object representing the Torch seed.
    """
    return pyrosetta_seed + (2 ** 31)


def atom_array_to_packed_pose(atom_array: AtomArray) -> PackedPose:
    """
    Convert a biotite `AtomArray` object to a PyRosetta `PackedPose`
    object in memory.

    Args:
        atom_array: A biotite `AtomArray` object.

    Returns:
        A `PackedPose` object.
    """
    buffer = StringIO()
    pdb = PDBFile()
    pdb.set_structure(atom_array)
    pdb.write(buffer)
    pdbstring = buffer.getvalue()
    packed_pose = io.pose_from_pdbstring(pdbstring)

    return packed_pose


def print_protocol_info(**kwargs: Any) -> None:
    """
    Print user-provided PyRosetta protocol and CUDA info during runtime.

    Keyword Args:
        PyRosettaCluster_*: Default `PyRosettaCluster` keyword arguments.

    Returns:
        None
    """
    protocol_name = kwargs["PyRosettaCluster_protocol_name"]
    protocol_number = kwargs["PyRosettaCluster_protocol_number"]
    seed = kwargs["PyRosettaCluster_seed"]
    client_repr = kwargs["PyRosettaCluster_client_repr"]
    cuda_is_available = torch.cuda.is_available()
    cuda_device_count = torch.cuda.device_count()
    cuda_device_name = torch.cuda.get_device_name(0) if cuda_is_available else None
    print(
        "Running --",
        f"Protocol name: '{protocol_name}';",
        f"Protocol number: {protocol_number};",
        f"Protocol seed: {seed};",
        f"Client: '{client_repr}';",
        f"CUDA is available: {cuda_is_available};",
        f"CUDA device count: {cuda_device_count};",
        f"CUDA device name: {cuda_device_name};",
        sep=" ",
    )


def get_sha256_digest(checkpoint_file: Path, size: int = 1024 * 1024, verbose: bool = False) -> str:
    """
    Generate a SHA256 digest of a binary checkpoint file.

    Args:
        checkpoint_file: A required `Path` object for which to generate the SHA256 digest.

    Keyword Args:
        size: An `int` object representing the chuck size in bytes to use for iterating
            over the checkpoint file.
            Default: 1024 * 1024
        verbose: A `bool` object specifying whether or not to print the result.

    Returns:
        A `str` object representing the SHA256 digest.
    """
    h = hashlib.sha256()
    with checkpoint_file.open("rb") as f:
        while True:
            data = f.read(size)
            if not data:
                break
            h.update(data)
    digest = h.hexdigest()
    if verbose:
        print(f"Generated SHA256 digest for checkpoint file '{checkpoint_file}': {digest}")

    return digest


def get_dataframe_from_pickle(scorefile: Path) -> pd.DataFrame:
    """
    Return a `pandas.DataFrame` object from a pickled `pandas.DataFrame`-formatted scorefile.

    Args:
        scorefile: A required `Path` object to the `pandas.DataFrame`-formatted scorefile.

    Returns:
        A `pandas.DataFrame` object.
    """
    df = (
        pd.read_pickle(scorefile, compression="infer")
        .reset_index(drop=False)
        .rename(columns={"index": "output_file"})
    )
    if set(df.columns) == {"output_file", "scores", "metadata", "instance"}:
        scores_df = df["scores"].apply(pd.Series)
        instance_df = df["instance"].apply(pd.Series)
        df = pd.concat([df[["output_file"]], instance_df[["decoy_ids", "seeds"]], scores_df], axis=1)

    return df


def get_lowest_scrmsd_decoy(scorefile: Path, verbose: bool = True) -> Path:
    """
    Get the lowest scRMSD decoy from a scorefile.

    Args:
        scorefile: A required `Path` object to the `pandas.DataFrame`-formatted scorefile.

    Keyword Args:
        verbose: A `bool` object specifying whether or not to print the result.
            Default: True

    Returns:
        A `Path` object representing the output decoy file in PDB format.
    """
    df = get_dataframe_from_pickle(scorefile)
    v = (
        df
        .loc[df["protocol_number"].eq(5)]
        .sort_values("bb_rmsd", ascending=True)
        .reset_index(drop=True)
        .iloc[0] # Top ranked design
    )
    output_file = Path(v["output_file"])
    if verbose:
        bb_rmsd = v["bb_rmsd"]
        total_score = v["total_score"]
        protocol_number = v["protocol_number"]
        decoy_ids = v["decoy_ids"]
        seeds = v["seeds"]
        metrics_str = "; ".join(
            [
                f"bb_rmsd={bb_rmsd}",
                f"total_score={total_score}",
                f"protocol_number={protocol_number}",
                f"decoy_ids={decoy_ids}",
                f"seeds={seeds}",
            ]
        )
        print(f"Lowest scRMSD decoy ({metrics_str}): {output_file}")

    return output_file


def get_bb_rmsd_nosuper(file1: str, file2: str, flags: Optional[str] = None) -> float:
    """
    Return the backbone heavy atom root-mean-squared deviation (RMSD) without superposition
    between two input structure files. If PyRosetta is not yet initialized, then PyRosetta
    will first be initialized with the optionally input PyRosetta initialization flags,
    otherwise "-mute all" flags are used.

    Args:
        file1: A `str` object representing the first structure file path.
        file2: A `str` object representing the second structure file path.

    Keyword Args:
        flags: An optional `str` object representing PyRosetta initialization
            options to use if PyRosetta is not already initialized.
            Default: ""

    Returns:
        A `float` object representing the backbone heavy atom RMSD.
    """
    from pyrosetta.rosetta.core.scoring import (
        rms_at_corresponding_atoms_no_super,
        setup_matching_protein_backbone_heavy_atoms,
    )
    from pyrosetta.rosetta.std import map_core_id_AtomID_core_id_AtomID

    for file in (file1, file2):
        if not isinstance(file, str) or not os.path.isfile(file):
            raise ValueError(f"The input file must be a `str` object and exist: '{file}'.")
    if flags and not isinstance(flags, str):
        raise ValueError(
            "The 'flags' keyword argument parameter must be an instance of `str`. "
            f"Received: {type(flags)}"
        )
    if not pyrosetta.rosetta.basic.was_init_called():
        extra_options = flags if flags else "-mute all"
        pyrosetta.init(options="", extra_options=extra_options, silent=True)

    pose1 = io.pose_from_file(file1).pose
    pose2 = io.pose_from_file(file2).pose
    atom_id_map = map_core_id_AtomID_core_id_AtomID()
    setup_matching_protein_backbone_heavy_atoms(pose1=pose1, pose2=pose2, atom_id_map=atom_id_map)
    bb_rmsd = rms_at_corresponding_atoms_no_super(mod_pose=pose1, ref_pose=pose2, atom_id_map=atom_id_map)

    return bb_rmsd


def get_sequence_percent_identity(seq1: str, seq2: str) -> float:
    """
    Return the sequence percent identity between two sequences.

    Args:
        seq1: A `str` object representing the first sequence.
        seq2: A `str` object representing the second sequence.

    Returns:
        A `float` object representing the sequence percent identity.
    """
    if len(seq1) != len(seq2):
        raise ValueError(f"Input sequences must be equal length: {len(seq1)} != {len(seq2)}")
    identical = sum(res1 == res2 for res1, res2 in zip(seq1, seq2))
    total = len(seq1)

    return (identical / total) * 100

In [ ]:
%%writefile {git_repo}/my_protocols.py
# @title Write the user-defined PyRosetta protocols

from pathlib import Path
from pyrosetta.distributed.cluster import requires_packed_pose
from pyrosetta.distributed.packed_pose.core import PackedPose
from typing import Any, Dict, List, Union

from my_utils import (
    atom_array_to_packed_pose,
    print_protocol_info,
    pyrosetta_to_torch_seed,
    timeit,
)


@timeit
def rfd3(packed_pose: PackedPose, **kwargs: Any) -> List[PackedPose]:
    """
    A PyRosetta protocol that runs RFdiffusion-3.

    Args:
        packed_pose: A required `None` object.

    Keyword Args:
        scorefxn_name: A required `str` object representing a score function name.
        cuda_visible_devices: A required key name for the 'CUDA_VISIBLE_DEVICES' environment variable parameter.
        PyRosettaCluster_*: Default `PyRosettaCluster` keyword arguments.

    Returns:
        A list of `PackedPose` objects.
    """
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = kwargs["cuda_visible_devices"]
    if os.getenv("CUDA_VISIBLE_DEVICES"):
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
        os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "1"

    import torch
    torch.use_deterministic_algorithms(True, warn_only=True)
    if os.getenv("CUDA_VISIBLE_DEVICES"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

    import pyrosetta.distributed.io as io

    from contextlib import nullcontext
    from lightning.fabric import seed_everything
    from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

    if packed_pose is not None:
        raise ValueError(
            f"The 'packed_pose' argument parameter must be `None`. Received: {type(packed_pose)}"
        )
    # Print runtime info
    print_protocol_info(**kwargs)
    # Setup seed
    torch_seed = pyrosetta_to_torch_seed(kwargs["PyRosettaCluster_seed"])
    # Configure RFD3
    config = RFD3InferenceConfig(
        specification={
            "length": kwargs["rfd3"]["length"],
        },
        diffusion_batch_size=kwargs["rfd3"]["diffusion_batch_size"],
        num_nodes=1,
        devices_per_node=1,
        verbose=True,
        seed=torch_seed,
    )
    # Initialize RFD3 inference engine
    model = RFD3InferenceEngine(**config)
    # Run RFD3
    with torch.amp.autocast("cuda", enabled=False) if os.getenv("CUDA_VISIBLE_DEVICES") else nullcontext():
        results = model.run(
            inputs=None,
            out_dir=None,
            n_batches=kwargs["rfd3"]["n_batches"],
        )
    # Update scores
    score_task = io.create_score_function(kwargs["scorefxn_name"])
    packed_poses = []
    for _example_id, rfd3_outputs in results.items():
        for rfd3_output in rfd3_outputs:
            packed_pose = score_task(atom_array_to_packed_pose(rfd3_output.atom_array))
            packed_pose = packed_pose.update_scores(
                rfd3_output_metadata=rfd3_output.metadata,
                sequence=packed_pose.pose.sequence(),
                protocol_number=kwargs["PyRosettaCluster_protocol_number"],
            )
            packed_poses.append(packed_pose)

    return packed_poses


@timeit
@requires_packed_pose
def proteinmpnn(packed_pose: PackedPose, **kwargs: Any) -> List[Union[PackedPose, Dict[str, Any]]]:
    """
    A PyRosetta protocol that runs ProteinMPNN.

    Args:
        packed_pose: A required input `PackedPose` object.

    Keyword Args:
        scorefxn_name: A required `str` object representing a score function name.
        cuda_visible_devices: A required key name for the 'CUDA_VISIBLE_DEVICES' environment variable parameter.
        PyRosettaCluster_*: Default `PyRosettaCluster` keyword arguments.

    Returns:
        A list of `PackedPose` objects with a `PyRosettaCluster` keyword arguments dictionary.
    """
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = kwargs["cuda_visible_devices"]
    if os.getenv("CUDA_VISIBLE_DEVICES"):
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
        os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "1"

    import torch
    torch.use_deterministic_algorithms(True, warn_only=True)
    if os.getenv("CUDA_VISIBLE_DEVICES"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

    import base64
    import pyrosetta
    import pyrosetta.distributed.io as io
    import toolz

    from contextlib import nullcontext
    from lightning.fabric import seed_everything
    from mpnn.inference_engines.mpnn import MPNNInferenceEngine

    # Print runtime info
    print_protocol_info(**kwargs)
    # Setup seed
    torch_seed = pyrosetta_to_torch_seed(kwargs["PyRosettaCluster_seed"])
    seed_everything(torch_seed)
    # Configure MPNNInferenceEngine
    config = {
        "model_type": "protein_mpnn",
        "is_legacy_weights": True,
        "out_directory": None,
        "write_structures": False,
        "write_fasta": False,
        "device": "cuda" if os.getenv("CUDA_VISIBLE_DEVICES") else "cpu",
    }
    # Configure per-input inference
    structure_path = Path(kwargs["PyRosettaCluster_tmp_path"]) / "tmp.pdb"
    pose = packed_pose.pose
    pose.cache.clear() # Clear scores from saved
    mpnn_pdbstring = io.to_pdbstring(pose)
    structure_path.write_text(mpnn_pdbstring)
    input_dicts = [
        {
            "structure_path": str(structure_path),
            "batch_size": kwargs["proteinmpnn"]["batch_size"],
            "number_of_batches": kwargs["proteinmpnn"]["number_of_batches"],
            "temperature": kwargs["proteinmpnn"]["temperature"],
            "structure_noise":  kwargs["proteinmpnn"]["structure_noise"],
            "omit": ["CYS", "UNK"],
            "decode_type": "auto_regressive",
            "causality_pattern": "auto_regressive",
            "remove_waters": True,
            "seed": torch_seed,
        }
    ]
    # Run ProteinMPNN
    model = MPNNInferenceEngine(**config)
    with torch.amp.autocast("cuda", enabled=False) if os.getenv("CUDA_VISIBLE_DEVICES") else nullcontext():
        results = model.run(input_dicts=input_dicts)
    # Update scores
    score_task = io.create_score_function(kwargs["scorefxn_name"])
    _reserved = pyrosetta.Pose().cache._reserved
    packed_poses = []
    for mpnn_output in results:
        _packed_pose = score_task(atom_array_to_packed_pose(mpnn_output.atom_array))
        _packed_pose = _packed_pose.update_scores(
            toolz.keyfilter(lambda k: k not in _reserved, packed_pose.pose.cache),
            mpnn_input_dict=mpnn_output.input_dict,
            mpnn_output_dict=mpnn_output.output_dict,
            mpnn_pdbstring=base64.b64encode(mpnn_pdbstring.encode("utf-8")),
            sequence=_packed_pose.pose.sequence(),
            protocol_number=kwargs["PyRosettaCluster_protocol_number"],
        )
        packed_poses.append(_packed_pose)
    # Cache ProteinMPNN input structure
    kwargs["mpnn_packed_pose"] = packed_pose.clone()

    return packed_poses + [kwargs]


@timeit
@requires_packed_pose
def rf3(packed_pose: PackedPose, **kwargs: Any) -> PackedPose:
    """
    A PyRosetta protocol that runs RosettaFold-3.

    Args:
        packed_pose: A required input `PackedPose` object.

    Keyword Args:
        scorefxn_name: A required `str` object representing a score function name.
        cuda_visible_devices: A required key name for the 'CUDA_VISIBLE_DEVICES' environment variable parameter.
        PyRosettaCluster_*: Default `PyRosettaCluster` keyword arguments.

    Returns:
        A `PackedPose` object.
    """
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = kwargs["cuda_visible_devices"]
    if os.getenv("CUDA_VISIBLE_DEVICES"):
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
        os.environ["CUDA_DEVICE_MAX_CONNECTIONS"] = "1"

    import torch
    torch.use_deterministic_algorithms(True, warn_only=True)
    if os.getenv("CUDA_VISIBLE_DEVICES"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

    import biotite.structure as struc
    import numpy as np
    import pyrosetta
    import pyrosetta.distributed.io as io
    import toolz

    from contextlib import nullcontext
    from rf3.inference_engines.rf3 import RF3InferenceEngine
    from rf3.utils.inference import InferenceInput

    # Print runtime info
    print_protocol_info(**kwargs)
    # Setup seed
    torch_seed = pyrosetta_to_torch_seed(kwargs["PyRosettaCluster_seed"])
    # Initialize RF3 inference engine
    engine = RF3InferenceEngine(
        n_recycles=kwargs["rf3"]["n_recycles"],
        diffusion_batch_size=kwargs["rf3"]["diffusion_batch_size"],
        num_steps=kwargs["rf3"]["num_steps"],
        template_noise_scale=1e-5,
        raise_if_missing_msa_for_protein_of_length_n=None,
        compress_outputs=False,
        early_stopping_plddt_threshold=None,
        metrics_cfg="default",
        ckpt_path="rf3",
        seed=torch_seed,
        num_nodes=1,
        devices_per_node=torch.cuda.device_count(),
        verbose=True,
    )
    # Dump temporary .pdb file
    tmp_path = Path(kwargs["PyRosettaCluster_tmp_path"])
    tmp_pdb_file = tmp_path / "tmp.pdb"
    tmp_pose = packed_pose.pose
    tmp_pose.cache.clear() # Clean up PDB file output to prevent error in `InferenceInput.from_cif_path`
    tmp_pdb_file.write_text(io.to_pdbstring(tmp_pose))
    # Setup RF3 inference inputs
    example_id = "rf3_example_id"
    inputs = InferenceInput.from_cif_path(
        path=tmp_pdb_file,
        example_id=example_id,
        template_selection=None,
        ground_truth_conformer_selection=None,
    )
    # Run RF3 inference
    with torch.amp.autocast("cuda", enabled=False) if os.getenv("CUDA_VISIBLE_DEVICES") else nullcontext():
        results = engine.run(
            inputs=inputs,
            out_dir=None,
            dump_predictions=False,
            dump_trajectories=False,
            one_model_per_file=True,
            annotate_b_factor_with_plddt=True,
            sharding_pattern=None,
            skip_existing=False,
            template_selection=None,
            ground_truth_conformer_selection=None,
            cyclic_chains=[],
        )
    rf3_output = results[example_id][0] # Top ranked prediction
    score_task = io.create_score_function(kwargs["scorefxn_name"])
    rf3_packed_pose = score_task(atom_array_to_packed_pose(rf3_output.atom_array))
    # Compute mean heavy-atom pLDDT per residue
    rf3_mean_plddt_per_res = [
        float(np.mean(res_atoms[res_atoms.element != "H"].get_annotation("b_factor")))
        for res_atoms in struc.residue_iter(rf3_output.atom_array)
    ]
    # Get backbone atom pLDDT per residue
    rf3_plddt_per_atom = {}
    for atom_name in ("N", "CA", "C", "O"):
        rf3_plddt_per_atom[atom_name] = [
            float(res_atoms[res_atoms.atom_name == atom_name].get_annotation("b_factor")[0])
            for res_atoms in struc.residue_iter(rf3_output.atom_array)
        ]
    # Update scores
    _reserved = pyrosetta.Pose().cache._reserved
    rf3_packed_pose = rf3_packed_pose.update_scores(
        toolz.keyfilter(lambda k: k not in _reserved, packed_pose.pose.cache),
        toolz.keymap(
            lambda k: f"rf3_{k}",
            toolz.merge(rf3_output.confidences, rf3_output.summary_confidences)
        ),
        rf3_example_id=rf3_output.example_id,
        rf3_sample_idx=rf3_output.sample_idx,
        rf3_seed=rf3_output.seed,
        rf3_mean_plddt_per_res=rf3_mean_plddt_per_res,
        rf3_plddt_per_atom=rf3_plddt_per_atom,
        sequence=rf3_packed_pose.pose.sequence(),
        protocol_number=kwargs["PyRosettaCluster_protocol_number"],
    )

    return rf3_packed_pose


def run_xml_str(packed_pose: PackedPose, xml_str: str) -> PackedPose:
    """
    Run the provided RosettaScripts XML string on the provided `PackedPose` object.

    Args:
        packed_pose: A required input `PackedPose` object.
        xml_str: A required RosettaScripts script as a `str` object.

    Returns:
        A `PackedPose` object.
    """
    import pyrosetta.distributed.io as io

    from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects

    xml_obj = XmlObjects.create_from_string(xml_str).get_mover("ParsedProtocol")
    pose = packed_pose.pose
    xml_obj.apply(pose)

    return io.to_packed(pose)


@timeit
@requires_packed_pose
def cst_cart_min_poly_gly(packed_pose: PackedPose, **kwargs: Any) -> PackedPose:
    """
    A PyRosetta protocol that converts the input `PackedPose` object to poly-glycine,
    and minimizes with C-alpha coordinate constraints.

    Args:
        packed_pose: A required input `PackedPose` object.

    Keyword Args:
        scorefxn_name: A required `str` object representing a score function name.
        PyRosettaCluster_*: Default `PyRosettaCluster` keyword arguments.

    Returns:
        A `PackedPose` object.
    """
    import pyrosetta.distributed.io as io

    from pathlib import Path

    # Print runtime info
    print_protocol_info(**kwargs)
    # Run RosettaScripts
    xml_str = """
    <ROSETTASCRIPTS>
        <SCOREFXNS>
            <ScoreFunction name="beta_cart" weights="beta_jan25_cart">
                <Reweight scoretype="coordinate_constraint" weight="1.0"/>
            </ScoreFunction>
        </SCOREFXNS>
        <MOVE_MAP_FACTORIES>
            <MoveMapFactory name="mmf" bb="1" chi="0" nu="0" branches="0" cartesian="1" jumps="0"/>
        </MOVE_MAP_FACTORIES>
        <MOVERS>
            <MakePolyX name="make_poly_gly"
                aa="GLY"
                keep_pro="0"
                keep_gly="0"
                keep_disulfide_cys="0"/>
            <VirtualRoot name="add_virtual_root" removable="1" remove="0"/>
            <VirtualRoot name="rm_virtual_root" removable="1" remove="1"/>
            <AddConstraints name="add_csts">
                <CoordinateConstraintGenerator name="coord_cst"
                    sd="0.25"
                    bounded="0"
                    bounded_width="0.0"
                    sidechain="0"
                    ca_only="1"
                    ambiguous_hnq="0"
                    native="0"
                    align_reference="0"/>
            </AddConstraints>
            <RemoveConstraints name="rm_csts" constraint_generators="coord_cst"/>
            <MinMover name="min"
                scorefxn="beta_cart"
                max_iter="500"
                type="lbfgs_armijo_nonmonotone"
                tolerance="0.01"
                movemap_factory="mmf"/>
        </MOVERS>
        <PROTOCOLS>
            <Add mover="make_poly_gly"/>
            <Add mover="add_virtual_root"/>
            <Add mover="add_csts"/>
            <Add mover="min"/>
            <Add mover="rm_csts"/>
            <Add mover="rm_virtual_root"/>
        </PROTOCOLS>
    </ROSETTASCRIPTS>
    """
    packed_pose = run_xml_str(packed_pose, xml_str)
    packed_pose = packed_pose.update_scores(
        sequence=packed_pose.pose.sequence(),
        protocol_number=kwargs["PyRosettaCluster_protocol_number"],
    )
    score_task = io.create_score_function(kwargs["scorefxn_name"])

    return score_task(packed_pose)


@timeit
@requires_packed_pose
def cart_min(packed_pose: PackedPose, **kwargs: Any) -> PackedPose:
    """
    A PyRosetta protocol that performs Cartesian minimization on the input `PackedPose` object.

    Args:
        packed_pose: A required input `PackedPose` object.

    Keyword Args:
        scorefxn_name: A required `str` object representing a score function name.
        PyRosettaCluster_*: Default `PyRosettaCluster` keyword arguments.

    Returns:
        A `PackedPose` object.
    """
    import pyrosetta.distributed.io as io

    from pathlib import Path

    # Print runtime info
    print_protocol_info(**kwargs)
    # Run RosettaScripts
    xml_str = """
    <ROSETTASCRIPTS>
        <SCOREFXNS>
            <ScoreFunction name="beta_cart" weights="beta_jan25_cart"/>
        </SCOREFXNS>
        <MOVE_MAP_FACTORIES>
            <MoveMapFactory name="mmf" bb="1" chi="1" nu="0" branches="0" cartesian="1" jumps="0"/>
        </MOVE_MAP_FACTORIES>
        <MOVERS>
            <MinMover name="min"
                scorefxn="beta_cart"
                max_iter="2000"
                type="lbfgs_armijo_nonmonotone"
                tolerance="0.0001"
                movemap_factory="mmf"/>
        </MOVERS>
        <PROTOCOLS>
            <Add mover="min"/>
        </PROTOCOLS>
    </ROSETTASCRIPTS>
    """
    packed_pose = run_xml_str(packed_pose, xml_str)
    packed_pose = packed_pose.update_scores(
        sequence=packed_pose.pose.sequence(),
        protocol_number=kwargs["PyRosettaCluster_protocol_number"],
    )
    score_task = io.create_score_function(kwargs["scorefxn_name"])

    return score_task(packed_pose)


@timeit
@requires_packed_pose
def compute_rmsd(packed_pose: PackedPose, **kwargs: Any) -> PackedPose:
    """
    A PyRosetta protocol that performs backbone heavy atom superposition and computes
    the backbone heavy atom root-mean-squared deviation (RMSD) between the input `PackedPose`
    and a reference `PackedPose` object.

    Args:
        packed_pose: A required input `PackedPose` object.

    Keyword Args:
        scorefxn_name: A required `str` object representing a score function name.
        mpnn_packed_pose: A required `PackedPose` object representing a reference structure.
        PyRosettaCluster_*: Default `PyRosettaCluster` keyword arguments.

    Returns:
        A `PackedPose` object.
    """
    import pyrosetta.distributed.io as io

    from pyrosetta.rosetta.core.scoring import (
        rms_at_corresponding_atoms,
        setup_matching_protein_backbone_heavy_atoms,
    )
    from pyrosetta.rosetta.std import map_core_id_AtomID_core_id_AtomID

    # Print runtime info
    print_protocol_info(**kwargs)
    # Setup protocol
    src_pose = packed_pose.pose
    ref_pose = kwargs["mpnn_packed_pose"].pose
    # Superimpose input onto reference and compute RMSD
    atom_id_map = map_core_id_AtomID_core_id_AtomID()
    setup_matching_protein_backbone_heavy_atoms(pose1=src_pose, pose2=ref_pose, atom_id_map=atom_id_map)
    bb_rmsd = rms_at_corresponding_atoms(mod_pose=src_pose, ref_pose=ref_pose, atom_id_map=atom_id_map)
    # Update scores
    packed_pose = packed_pose.update_scores(
        bb_rmsd=bb_rmsd,
        sequence=packed_pose.pose.sequence(),
        protocol_number=kwargs["PyRosettaCluster_protocol_number"],
    )
    score_task = io.create_score_function(kwargs["scorefxn_name"])

    return score_task(packed_pose)

In [ ]:
%%writefile {git_repo}/my_runner.py
# @title Write the original PyRosettaCluster simulation launch script

__author__ = "" # Optionally enter your name to add a license to the output results
__email__ = "" # Optionally enter your email to add to the output results

import argparse
import os
import pyrosetta
import subprocess
import sys
import torch

from collections.abc import Callable
from dask.distributed import Client, LocalCluster
from pathlib import Path
from pyrosetta.distributed.cluster import PyRosettaCluster, export_init_file
from typing import Any, Dict, Generator, Union

from my_protocols import cart_min, compute_rmsd, cst_cart_min_poly_gly, proteinmpnn, rf3, rfd3
from my_utils import get_lowest_scrmsd_decoy, get_sha256_digest


class Resources:
    """Manage compute resources for a PyRosettaCluster simulation."""
    _gpu_enabled_protocols: set[Callable[..., Any]] = {proteinmpnn, rf3, rfd3}
    _cpu_resource: Dict[str, Union[float, int]] = {"CPU": 1}
    _gpu_resource: Dict[str, Union[float, int]] = {"GPU": 1}

    def __init__(self, *protocols: Callable[..., Any], gpu: bool = False) -> None:
        for protocol in protocols:
            if not callable(protocol):
                raise ValueError(
                    f"The protocol '{protocol!r}' is not a callable object. Received: {type(protocol)}"
                )
        if not isinstance(gpu, bool):
            raise ValueError(
                f"The 'gpu' keyword argument must be of type `bool`. Received: {type(gpu)}"
            )
        self.protocols: tuple[Callable[..., Any]] = protocols
        self.gpu: bool = gpu

    def get(self) -> list[Dict[str, Union[float, int]]]:
        """
        Get resources for the `PyRosettaCluster.distribute(resources=...)` keyword argument.

        Returns:
            A `list` object of `dict` objects representing protocol-specific abstract resource constraints.
        """
        return [
            Resources._gpu_resource.copy()
            if self.gpu and protocol in Resources._gpu_enabled_protocols
            else Resources._cpu_resource.copy()
            for protocol in self.protocols
        ]


def initialize_pyrosetta() -> None:
    """
    Initialize PyRosetta on the client.

    Returns:
        None
    """
    pyrosetta.init("-run:constant_seed 1 -multithreading:total_threads 1")
    pyrosetta.secure_unpickle.add_secure_package("pandas")
    pyrosetta.secure_unpickle.add_secure_package("pyarrow")


def download_checkpoints() -> None:
    """
    Download Foundry model weights.

    Returns:
        None
    """
    subprocess.run(
        ["foundry", "install", "rfd3", "proteinmpnn", "rf3"],
        check=True,
    )


def get_system_info(gpu: bool) -> Dict[str, Any]:
    """
    Get system information for the PyRosettaCluster simulation.

    Args:
        gpu: A required `bool` object specifying whether or not to use GPU resources.

    Returns:
        A `dict` object representing the `PyRosettaCluster(system_info=...)` keyword argument parameter.
    """
    # Cache system platform
    system_info = {"sys.platform": sys.platform}
    # Cache CUDA device info
    _cuda_is_available = torch.cuda.is_available()
    if gpu and _cuda_is_available:
        system_info["torch.cuda.is_available()"] = _cuda_is_available
        _device_count = torch.cuda.device_count()
        system_info["torch.cuda.device_count()"] = _device_count
        for i in range(_device_count):
            system_info[f"torch.cuda.get_device_name({i})"] = torch.cuda.get_device_name(i)
    # Cache Foundry checkpoint file checksums
    system_info.setdefault("checkpoints", {})
    ckpt_dir = (Path(os.getenv("HOME")) / ".foundry" / "checkpoints").resolve()
    ckpt_files = ckpt_dir.glob("*")
    for ckpt_file in ckpt_files:
        system_info["checkpoints"][ckpt_file.name] = get_sha256_digest(ckpt_file, verbose=True)

    return system_info


def get_cuda_visible_devices(gpu: bool) -> str:
    """
    Return the 'CUDA_VISIBLE_DEVICES' environment variable value for the PyRosettaCluster simulation.

    Args:
        gpu: A required `bool` object specifying whether or not to use GPU resources.

    Returns:
        A `str` object representing the 'CUDA_VISIBLE_DEVICES' environment variable value.
    """
    return (
        ",".join(map(str, range(torch.cuda.device_count())))
        if gpu and torch.cuda.is_available()
        else ""
    )


def create_tasks(num_tasks: int, gpu: bool) -> Generator[Dict[str, Any], None, None]:
    """
    Create tasks for a PyRosettaCluster simulation that uses a Dask `LocalCluster` instance.

    Args:
        num_tasks: A required `int` object representing the number of tasks to generate.
        gpu: A required `bool` object specifying whether or not to use GPU resources.

    Yields:
        An output `dict` object representing a task.
    """
    if not isinstance(num_tasks, int):
        raise ValueError(
            f"The 'num_tasks' keyword argument parameter must be of type `int`. Received: {type(num_tasks)}"
        )
    if not isinstance(gpu, bool):
        raise ValueError(
            f"The 'gpu' keyword argument parameter must be of type `bool`. Received: {type(gpu)}"
        )

    cuda_visible_devices = get_cuda_visible_devices(gpu)
    for i in range(num_tasks):
        yield {
            "options": {
                "beta_jan25": "1",
                "score:count_pair_hybrid": "0",
            },
            "extra_options": {
                "multithreading:total_threads": "1",
                "linmem_ig": "10",
                "no_optH": "0",
                "flip_HNQ": "0",
                "no_his_his_pairE": "1",
                "run:preserve_header": "1",
                "nblist_autoupdate": "1",
                "write_all_connect_info": "1",
                "connect_info_cutoff": "3.0",
            },
            "set_logging_handler": "logging",
            # RFdiffusion-3 parameters
            "rfd3": {
                "length": "30",
                "diffusion_batch_size": 2, # Increase for more backbones
                "n_batches": 1,
            },
            # ProteinMPNN parameters
            "proteinmpnn": {
                "temperature": 0.1,
                "structure_noise": 0.0,
                "batch_size": 2, # Increase for more sequences per backbone
                "number_of_batches": 1,
            },
            # RoseTTAFold-3 parameters
            "rf3": {
                "diffusion_batch_size": 2, # Increase for more structure predictions per sequence
                "n_recycles": 5,
                "num_steps": 50,
            },
            # Protocol-specific parameters
            "cuda_visible_devices": cuda_visible_devices,
            "scorefxn_name": "beta_jan25",
            # Metadata
            "task_idx": i,
        }


def main(
    output_path: str,
    scratch_dir: str,
    output_init_file: str,
    num_tasks: int,
    gpu: bool,
) -> None:
    """Run the PyRosettaCluster simulation."""
    # Initialize PyRosetta
    initialize_pyrosetta()
    # Download Foundry checkpoints
    download_checkpoints()

    # Set the number of workers dynamically
    n_workers = 1
    print(f"Spinning up {n_workers} Dask workers!")

    # Setup client resources
    resources = {}
    resources.update(Resources._cpu_resource)
    if gpu:
        resources.update(Resources._gpu_resource)

    # Run the simulation
    with LocalCluster(
        n_workers=n_workers,
        threads_per_worker=2,
        memory_limit=f"{12.7 / n_workers:.2f}GB",
        scheduler_port=8786,
        dashboard_address=":8787",
        resources=resources,
    ) as cluster, Client(cluster) as client:
        # Setup protocols
        protocols = [rfd3, cst_cart_min_poly_gly, proteinmpnn, rf3, cart_min, compute_rmsd]
        num_protocols = len(protocols)
        PyRosettaCluster(
            tasks=create_tasks(num_tasks, gpu),
            input_packed_pose=None,
            client=client,
            scratch_dir=scratch_dir,
            output_path=output_path,
            project_name="pyrosettacluster-foundry-tutorial",
            simulation_name=f"gpu-{int(gpu)}",
            simulation_records_in_scorefile=True,
            filter_results=True,
            compression=True,
            compressed=False,
            output_decoy_types=[".pdb", ".b64_pose"],
            output_scorefile_types=[".json", ".bz2"],
            save_all=True,
            system_info=get_system_info(gpu),
            author=__author__,
            email=__email__,
            license=(
                f"Copyright (c) {__author__}. "
                "Creative Commons Attribution 4.0 International (CC BY)" # Optionally change the license
            ) if __author__ else "",
        ).distribute(
            protocols=protocols,
            clients_indices=[0] * num_protocols,
            priorities=list(range(num_protocols)),
            resources=Resources(*protocols, gpu=gpu).get(),
        )

    # Export PyRosetta initialization file with decoy of interest while PyRosetta is still initialized
    scorefile = Path(output_path) / "scores.bz2"
    pdb_output_file = get_lowest_scrmsd_decoy(scorefile) # Decoy of interest in PDB file format
    b64_output_file = pdb_output_file.with_suffix(".b64_pose") # Base64-encoded picked Pose file with full atomic coordinate precision
    export_init_file(
        str(b64_output_file),
        output_init_file=output_init_file,
        compressed=None,
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Launch PyRosettaCluster.",
    )
    parser.add_argument(
        "--output_path",
        type=str,
        required=True,
        help="The PyRosettaCluster simulation output directory.",
    )
    parser.add_argument(
        "--scratch_dir",
        type=str,
        required=True,
        help="The PyRosettaCluster simulation scratch directory.",
    )
    parser.add_argument(
        "--output_init_file",
        type=str,
        required=True,
        help="The output PyRosetta initialization file path that will contain the decoy of interest.",
    )
    parser.add_argument(
        "--num_tasks",
        type=int,
        default=1,
        required=False,
        help="The number of tasks in the PyRosettaCluster simulation.",
    )
    parser.add_argument(
        "--gpu",
        action=argparse.BooleanOptionalAction,
        default=False,
        help="Run the PyRosettaCluster simulation with GPUs enabled or disabled.",
    )

    args = parser.parse_args()
    main(
        args.output_path,
        args.scratch_dir,
        args.output_init_file,
        args.num_tasks,
        args.gpu,
    )

In [ ]:
# @title Before launching the PyRosettaCluster simulation, commit the user-defined PyRosetta protocols to the Git repository for scientific reproducibility
if not os.getenv("DEBUG"):
    !cd {git_repo} && \
        git add . && \
        git commit -m "Initial commit of my PyRosetta protocols"

##### *Note:* in a real experiment, you may wish to push the Git repository to a remote GitHub repository using:
```
git remote add origin <remote-repository-url>
git push -u origin main
```

##### Now we are ready to launch the original PyRosettaCluster simulation!

##### For further details, see: https://github.com/RosettaCommons/pyrosetta-extras/tree/main/pyrosettacluster#running-pyrosettacluster-simulations

In [ ]:
# @title Launch the original PyRosettaCluster simulation, and export the PyRosetta initialization file
if not os.getenv("DEBUG"):
    from torch.cuda import is_available

    original_gpu_flag = "--gpu" if is_available() else "--no-gpu"
    original_scratch_dir = original_output_path / "scratch"
    output_init_file = git_repo / "my_decoy.init"
    !export $(xargs < {original_env_file}) && \
        cd {git_repo} && \
        pixi run --manifest-path {original_manifest_path} \
        python -m my_runner \
        {original_gpu_flag} \
        --output_path {original_output_path} \
        --scratch_dir {original_scratch_dir} \
        --output_init_file {output_init_file} \
        --num_tasks 1

In [ ]:
# @title After running the original PyRosettaCluster simulation, commit the decoy of interest to the Git repository
# Note: for real experiments, you may wish to commit all data to the Git repository
if not os.getenv("DEBUG"):
    !cd {git_repo} && \
        git add . && \
        git commit -m "Initial commit of my decoy of interest"

##### *Note:* for real experiments, you may wish to push the decoy of interest to a remote GitHub repository using `git push`. Additionally, if uploading results to a public repository, it may be preferrable to push the output decoy of interest in *PDB file format* or the output scorefile in *JSON file format*; because these are plain text formats, they are auditable by independent investigators. Note that the PyRosetta initialization file contains a Base64-encoded pickled `Pose` object representing the output decoy, which may seem risky to unpickle by independent investigators. This tutorial presumes that the exported PyRosetta initialization file is shared within a trusted organization.

# Stage II: Reproduction Simulation

##### For the remainder of the notebook, we demonstrate reproducing the output decoy of interest from the original simulation as if running on a separate filesystem. Therefore, we will perform some steps that appear unnecessary within the same Jupyter notebook, but will make sense from the perspective of a colleague reproducing the original simulation.

In [ ]:
# @title Enter the `PyRosettaCluster` output directory for the reproduction simulation

# @markdown ##### E.g., enter `/content/reproduce_data`. If the path begins with `/content/drive/MyDrive`, you will be asked to mount Google Drive
reproduce_output_path = "/content/reproduce_data" # @param {type:"string"}
reproduce_output_path = Path(reproduce_output_path)

if not os.getenv("DEBUG"):
    if str(reproduce_output_path).startswith("/content/drive/MyDrive"):
        from google.colab import drive
        drive.mount("/content/drive")

    try:
        reproduce_output_path.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        print(f"Please remove the output directory and try again: {reproduce_output_path}")

##### The exported PyRosetta initialization file contains a Base64-encoded pickled `Pose` object which contains the original environment file string (with Foundry, etc., as a `pixi.toml` file). Therefore, we must read the PyRosetta build signature embedded in the PyRosetta initialization file, and install a matching version of PyRosetta to properly extract the original environment file string. Only PyRosetta and a compatible version of the `pyrosetta-distributed` meta-package are needed at this step. *Note:* we already have the original Pixi project installed earlier in this notebook – this step is only necessary as if we were reproducing the simulation on a separate filesystem.

##### For further details, see: https://github.com/RosettaCommons/pyrosetta-extras/tree/main/pyrosettacluster#extract-environment-configuration

In [ ]:
# @title Create a temporary Pixi project solely for environment extraction
def get_versions(init_file):
    """Progammatically read the Python and PyRosetta version from a PyRosetta initialization file."""
    build = !jq -r '.pyrosetta_build' {init_file}
    py = re.search(r"python(\d)(\d{2})", build[0])
    pr = re.search(r"\](\d{4})\.(\d{2})", build[0])
    return f"{py[1]}.{py[2]}", f"{pr[1]}.{int(pr[2])}"

if not os.getenv("DEBUG"):
    # Install Pixi
    if not shutil.which("pixi"):
        !export PIXI_VERSION={PIXI_VERSION} && curl -fsSL https://pixi.sh/install.sh | sh
        os.environ["PATH"] = f"{os.getenv('PATH')}{os.pathsep}/root/.pixi/bin"
    !pixi --version

    # Setup Pixi project
    rosettacommons_conda_channel = "https://conda.rosettacommons.org"
    python_version, pyrosetta_version = get_versions(output_init_file)
    tmp_project_name = "extract_environment"
    tmp_manifest_path = Path.cwd() / tmp_project_name / "pixi.toml"
    if not tmp_manifest_path.exists():
        !pixi init {tmp_project_name}
        !pixi workspace --manifest-path {tmp_manifest_path} --no-progress channel add --prepend {rosettacommons_conda_channel}
        !pixi add --manifest-path {tmp_manifest_path} --no-progress python={python_version} pyrosetta={pyrosetta_version} # Install original PyRosetta version
        !pixi add --manifest-path {tmp_manifest_path} --no-progress --pypi pyrosetta-distributed>=0.0.5 # Needed for imports only, otherwise not used
    if (tmp_manifest_path.parent / "pixi.lock").exists():
        !pixi install --manifest-path {tmp_manifest_path} --frozen --no-progress
    else:
        !pixi install --manifest-path {tmp_manifest_path} --no-progress

    # Ensure original PyRosetta build signature is reproduced
    original_build_signature = !jq -r '.pyrosetta_build' {output_init_file}
    tmp_build_signature = !pixi run --manifest-path {tmp_manifest_path} python -c "import pyrosetta; print(pyrosetta._build_signature())"
    ansi_escape = re.compile(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])')
    if ansi_escape.sub("", tmp_build_signature[-1]) != original_build_signature[-1]:
        raise ValueError(
            "PyRosetta build signature must match the original to extract the environment "
            "file string from a PyRosetta initialization file from PyRosettaCluster!"
        )

In [ ]:
# @title Clone the *RosettaCommons/pyrosetta-extras* GitHub repository
if not os.getenv("DEBUG"):
    extras_repo_path = Path.cwd() / "pyrosetta-extras"
    if not extras_repo_path.is_dir():
        !git clone https://github.com/RosettaCommons/pyrosetta-extras.git {extras_repo_path}
    else:
        !cd {extras_repo_path} && git pull --rebase # Pull latest commit if already cloned

In [ ]:
# @title Extract the environment file string from a PyRosettaCluster output decoy or scorefile
if not os.getenv("DEBUG"):
    env_dir = reproduce_output_path / "reproduce_env"
    dump_env_file_module = extras_repo_path / "pyrosettacluster" / "dump_env_file.py"
    dump_env_outputs = !pixi run --manifest-path {tmp_manifest_path} \
        python {dump_env_file_module} \
        --input_file {output_init_file} \
        --env_dir {env_dir}

    # The original Git commit SHA-1 hash we created prior to running the original PyRosettaCluster simulation
    # (containing the user-defined PyRosetta protocols) is conveniently printed when extracting the
    # environment file string. Here, we programmatically parse it from the printed outputs
    sha1 = next(iter(filter(lambda v: v.startswith("[INFO] GitHub SHA1: "), dump_env_outputs))).split()[-1]

##### Now that we have extracted the original environment file string from the exported PyRosetta initialization file, we can recreate the original Pixi project, which is needed to reproduce the output decoy of interest.

##### For further details, see: https://github.com/RosettaCommons/pyrosetta-extras/tree/main/pyrosettacluster#recreate-environment

In [ ]:
# @title Recreate the original Pixi project

# Specify the environment manager to use to recreate the environment. If not specified,
# the environment manager is auto-detected from having either `conda`, `mamba`, `uv` or `pixi`
# command-line executables in one of the directories on the operating system's `PATH`
# environment variable. Below, we specify `pixi` for the sake of being explicit.
os.environ["PYROSETTACLUSTER_ENVIRONMENT_MANAGER"] = "pixi"

# Note: we use the **system Python executable** to recreate the Pixi project
# since we cannot create a Pixi project from within the temporary Pixi project
if not os.getenv("DEBUG"):
    recreate_env_module = extras_repo_path / "pyrosettacluster" / "recreate_env.py"
    !python {recreate_env_module} --env_dir {env_dir}

##### We also now checkout the Git repository at the original Git commit SHA-1, which is needed to reproduce the output decoy of interest since it contains the user-defined PyRosetta protocol definitions. In a real experiment, you would normally have to clone the repository from the remote GitHub repository, which requires knowing the organization/user and the repository name (if not, don't worry, there are ways to search GitHub by commit SHA-1).

##### For further details, see: https://github.com/RosettaCommons/pyrosetta-extras/tree/main/pyrosettacluster#clone-original-repository

In [ ]:
# @title Checkout the Git repository at the original Git commit SHA-1
if not os.getenv("DEBUG"):
    cloned_git_repo = reproduce_output_path / "original_data"
    !git clone {git_repo} {cloned_git_repo}
    !cd {cloned_git_repo} && git checkout {sha1}
else:
    cloned_git_repo = Path.cwd() / "my_cloned_git_repo"
    cloned_git_repo.mkdir(parents=True, exist_ok=False)

In [ ]:
%%writefile {cloned_git_repo}/reproduce.py
# @title Write the reproduction PyRosettaCluster simulation launch script

import argparse
import pyrosetta
import pyrosetta.distributed.io as io

from dask.distributed import LocalCluster, Client
from pyrosetta.distributed.cluster import get_scores_dict, reproduce

from my_protocols import cart_min, compute_rmsd, cst_cart_min_poly_gly, proteinmpnn, rf3, rfd3
from my_runner import Resources, download_checkpoints, get_system_info, initialize_pyrosetta
from my_utils import get_sha256_digest


def verify_system_info(original_system_info, reproduce_system_info):
    """
    Verify that the extra simulation information cataloged in the PyRosettaCluster
    system information dictionary is identical for scientific reproducibility purposes.
    """
    for k in original_system_info:
        reproduce_val = reproduce_system_info.get(k)
        original_val = original_system_info.get(k)
        if k == "checkpoints":
            for ckpt_file in original_val:
                if ckpt_file not in reproduce_val:
                    raise FileNotFoundError(f"The original checkpoint file '{ckpt_file}' was not downloaded.")
                if reproduce_val.get(ckpt_file) != original_val.get(ckpt_file):
                    raise ValueError(f"Checksums differ for checkpoint file '{ckpt_file}'.")
        else:
            if reproduce_val != original_val:
                raise ValueError(f"Original info '{original_val}' is not identical to current: '{reproduce_val}'.")


def main(
    input_file: str,
    output_path: str,
    scratch_dir: str,
    gpu: bool,
):
    # Setup reproduction simulation like the original
    if input_file.endswith((".init", ".init.bz2")):
        io.init_from_file(input_file)
        pyrosetta.secure_unpickle.add_secure_package("pandas")
        pyrosetta.secure_unpickle.add_secure_package("pyarrow")
    else:
        initialize_pyrosetta()
    download_checkpoints()
    system_info = get_system_info(gpu)

    # Verify checkpoint checksums
    original_system_info = get_scores_dict(input_file)["instance"]["system_info"]
    verify_system_info(original_system_info, system_info)

    # Run reproduction simulation
    n_workers = 1
    with LocalCluster(
        n_workers=n_workers,
        threads_per_worker=2,
        memory_limit=f"{12.7 / n_workers:.2f}GB",
        scheduler_port=8786,
        dashboard_address=":8787",
    ) as cluster, Client(cluster) as client:
        reproduce(
            input_file=input_file,
            protocols=None, # Auto-detect imported protocol(s)
            client=client,
            input_packed_pose=None,
            instance_kwargs={
                "output_path": output_path,
                "scratch_dir": scratch_dir,
                "project_name": "pyrosettacluster-foundry-tutorial",
                "simulation_name": f"reproduce_gpu-{int(gpu)}",
                "system_info": system_info,
                "output_decoy_types": [".pdb", ".b64_pose", ".init"], # Write output `.init` file
            },
            resources=None,
            skip_corrections=False,
            init_from_file_kwargs=None,
        )


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_file", type=str, required=True)
    parser.add_argument("--output_path", type=str, required=True)
    parser.add_argument("--scratch_dir", type=str, required=True)
    parser.add_argument("--gpu", action=argparse.BooleanOptionalAction, default=False)
    args = parser.parse_args()
    main(
        args.input_file,
        args.output_path,
        args.scratch_dir,
        args.gpu,
    )

##### Now we are ready to reproduce the original PyRosettaCluster simulation!

##### For further details, see: https://github.com/RosettaCommons/pyrosetta-extras/tree/main/pyrosettacluster#run-reproduction-script

In [ ]:
# @title 🚀 Reproduce the PyRosettaCluster simulation
if not os.getenv("DEBUG"):
    from torch.cuda import is_available

    env_dir_manifest_path = env_dir / "pixi.toml"
    reproduce_scratch_dir = reproduce_output_path / "scratch"
    reproduce_env_file = reproduce_output_path / ".env"
    reproduce_env_file.write_text(
        "\n".join(
            [
                "PDB_MIRROR_PATH=",
                "CCD_MIRROR_PATH=",
                "LOCAL_MSA_DIRS=",
                "HBPLUS_PATH=",
                "X3DNA_PATH=",
                "DSSP_PATH=",
                "HHFILTER_PATH=",
                "MMSEQS2_PATH=",
                "COLABFOLD_LOCAL_DB_PATH_GPU=",
                "COLABFOLD_LOCAL_DB_PATH_CPU=",
                "COLABFOLD_NET_DB_PATH_GPU=",
                "COLABFOLD_NET_DB_PATH_CPU=",
                "FOUNDRY_CHECKPOINT_DIRS=",
            ]
        )
    )

    reproduce_gpu_flag = "--gpu" if is_available() else "--no-gpu"
    !export $(xargs < {reproduce_env_file}) && \
        cd {cloned_git_repo} && \
        pixi run --manifest-path {env_dir_manifest_path} \
        python -m reproduce \
        {reproduce_gpu_flag} \
        --input_file {output_init_file} \
        --output_path {reproduce_output_path} \
        --scratch_dir {reproduce_scratch_dir}

##### 🎉 Congrats! You have now recreated a virtual environment and used it to successfully reproduce an output decoy from a distributed PyRosettaCluster simulation. *Note:* use of one or more GPUs with *Foundry* cannot guarantee reproducibility of the decoy of interest due to variability presumably caused by floating-point non-associativity and/or unordered parallel reductions.

# Stage III: Analysis (Optional)

##### For the remainder of the notebook, we analyze the original decoy of interest and the reproduced decoy of interest to confirm (or disprove) accurate decoy reproducibility.

#### *Warning:* if GPU devices were used for the original or reproduction simulation, the following unit test evaluation *will raise an error* (as expected due to variability presumably caused by floating-point non-associativity and/or unordered parallel reductions). See the *Challenge* at the start of the notebook to enable simulation reproducibility with GPU devices! You may still proceed in any case.

In [ ]:
%%writefile {cloned_git_repo}/evaluate.py
# @title Write the evaluation script

import argparse
import pyrosetta
import pyrosetta.distributed.io as io
import unittest

from pathlib import Path
from pyrosetta import Pose
from pyrosetta.distributed.cluster import PackedPoseHasher


class TestPoses(unittest.TestCase):
    """Unit test case for the identity of PyRosetta `Pose` objects' scientific states."""

    @classmethod
    def setUpClass(cls):
        """Set up the unit test class."""
        if (
            cls.original_output_file.endswith((".init", ".init.bz2"))
            and cls.reproduce_output_file.endswith((".init", ".init.bz2"))
        ):
            cls.original_pose = io.pose_from_init_file(cls.original_output_file).pose
            cls.reproduce_pose = io.pose_from_init_file(cls.reproduce_output_file).pose
        else:
            if not pyrosetta.rosetta.basic.was_init_called():
                pyrosetta.init(options="", extra_options=cls.pyrosetta_init_flags, silent=True)
            cls.original_pose = io.pose_from_file(cls.original_output_file).pose
            cls.reproduce_pose = io.pose_from_file(cls.reproduce_output_file).pose

    def assert_atom_coordinates(self, pose1: Pose, pose2: Pose) -> None:
        """Assert that two input `Pose` objects have identical atomic coordinates."""
        self.assertEqual(pose1.size(), pose2.size())
        for res in range(1, pose1.size() + 1):
            res1 = pose1.residue(res)
            res2 = pose2.residue(res)
            self.assertEqual(res1.name(), res2.name())
            self.assertEqual(res1.natoms(), res2.natoms())
            for atom in range(1, res1.natoms() + 1):
                self.assertEqual(res1.atom_name(atom), res2.atom_name(atom))
                for axis in "xyz":
                    self.assertEqual(
                        float(getattr(res1.atom(atom).xyz(), axis)),
                        float(getattr(res2.atom(atom).xyz(), axis)),
                    )

    def assert_rmsd(self, pose1: Pose, pose2: Pose) -> None:
        """Assert that two input `Pose` objects have an RMSD of zero without superposition."""
        self.assertEqual(pose1.size(), pose2.size())
        # Test RMSDs without superimposing
        self.assertEqual(
            pyrosetta.rosetta.core.scoring.all_atom_rmsd_nosuper(pose1, pose2),
            0.0,
        )
        self.assertEqual(
            pyrosetta.rosetta.core.scoring.all_scatom_rmsd_nosuper(pose1, pose2),
            0.0,
        )
        for res in range(1, pose1.size() + 1):
            v1 = pyrosetta.rosetta.utility.vector1_unsigned_long()
            v1.append(res)
            rmsd = pyrosetta.rosetta.core.scoring.all_atom_rmsd_nosuper(
                pose1=pose1,
                pose2=pose2,
                pose1_residues=v1,
                pose2_residues=v1,
            )
            self.assertEqual(rmsd, 0.0)

    def assert_total_score(self, pose1: Pose, pose2: Pose) -> None:
        """Assert that two input `Pose` objects have identical total energy."""
        scorefxn = pyrosetta.get_score_function()
        self.assertEqual(scorefxn(pose1.clone()), scorefxn(pose2.clone()))

    def assert_pose_digests(
        self,
        pose1: Pose,
        pose2: Pose,
        include_cache=True,
        include_comments=False,
    ) -> None:
        """Assert that two input `Pose` objects have identical digests."""
        self.assertEqual(
            PackedPoseHasher(
                packed_pose=pose1,
                include_cache=include_cache,
                include_comments=include_comments,
            ).digest(),
            PackedPoseHasher(
                packed_pose=pose2,
                include_cache=include_cache,
                include_comments=include_comments,
            ).digest(),
        )

    def test_poses(self) -> None:
        """Unit test for the identity of the scientific states of two `Pose` objects."""
        self.assert_atom_coordinates(self.original_pose, self.reproduce_pose)
        self.assert_rmsd(self.original_pose, self.reproduce_pose)
        self.assert_total_score(self.original_pose, self.reproduce_pose)
        self.assert_pose_digests(self.original_pose, self.reproduce_pose)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--original_output_file", type=Path, required=True)
    parser.add_argument("--reproduce_output_file", type=Path, required=True)
    parser.add_argument(
        "--pyrosetta_init_flags",
        type=str,
        required=False,
        default="-run:constant_seed 1 -out:level 200",
    )
    args, remaining_argv = parser.parse_known_args()
    if not args.original_output_file.is_file():
        parser.error(f"The file '{args.original_output_file}' does not exist.")
    if not args.reproduce_output_file.is_file():
        parser.error(f"The file '{args.reproduce_output_file}' does not exist.")
    # Inject args into the class before running test
    TestPoses.original_output_file = str(args.original_output_file)
    TestPoses.reproduce_output_file = str(args.reproduce_output_file)
    TestPoses.pyrosetta_init_flags = args.pyrosetta_init_flags
    # Run test
    unittest.main(argv=[__file__] + remaining_argv)

In [ ]:
# @title Validate that the original decoy of interest and the reproduced decoy of interest have identical scientific states
if not os.getenv("DEBUG"):
    if original_gpu_flag == "--gpu" or reproduce_gpu_flag == "--gpu":
        print("*Warning*: Cannot guarantee PyRosettaCluster simulation reproducibility with GPUs enabled!")

    def get_reproduce_output_file(output_path, protocol_number=5):
        with (output_path / "scores.json").open("r") as f:
            for line in f:
                d = json.loads(line)
                if d["scores"]["protocol_number"] == protocol_number:
                    return Path(d["metadata"]["output_file"]).with_suffix(".init") # Read output `.init` file

    reproduce_output_file = get_reproduce_output_file(reproduce_output_path, protocol_number=5)
    !cd {cloned_git_repo} && \
        pixi run --manifest-path {env_dir_manifest_path} \
        python -m evaluate \
        --original_output_file {output_init_file} \
        --reproduce_output_file {reproduce_output_file}

# Compare intermediate decoys from the original and reproduction PyRosettaCluster simulation

##### *Note:* the following analysis requires that `save_all` keyword argument was set to `True` during the original and reproduction PyRosettaCluster simulations, and that the original output scorefile and the reproduced output scorefile still exist in their default filesystem locations. We did not commit the original scorefile and all output decoys to the Git repository, so this next step is only optional as part of the tutorial, and not a step a colleague would be able to run without the full original dataset.

In [ ]:
%%writefile {cloned_git_repo}/compare.py
# @title Compare original and reproduced intermediate decoys

import argparse
import json
import pandas as pd

from pathlib import Path
from typing import Dict, List

from my_utils import (
    get_bb_rmsd_nosuper,
    get_dataframe_from_pickle,
    get_sequence_percent_identity,
)


def get_decoy_ids(df: pd.DataFrame, protocol_number: int) -> List[int]:
    return (
        df
        .loc[df["protocol_number"].eq(protocol_number)]
        .sort_values("bb_rmsd", ascending=True)
        .reset_index(drop=True)
        .iloc[0, df.columns.get_loc("decoy_ids")] # Top ranked design
    )


def assert_one_row(df: pd.DataFrame) -> pd.DataFrame:
    assert len(df) == 1, f"The `pd.DataFrame` object has more than one row: {len(df)}"
    return df


def print_data(protocol_number_data: Dict[int, Dict[str, float]]) -> None:
    print("Results:")
    for protocol_number, data in protocol_number_data.items():
        print(f"    Protocol number: {protocol_number}")
        for k, v in data.items():
            if k == "bb_rmsd":
                print(f"        Backbone Heavy-Atom RMSD (No Superposition) (Å): {v}")
            elif k == "sequence_percent_identity":
                print(f"        Sequence Percent Identity (%): {v}")
            elif k == "delta_total_score":
                print(f"        ∆Total Score (REU): {v}")


def write_data(output_json_file: Path, protocol_number_data: Dict[int, Dict[str, float]]) -> None:
    with output_json_file.open("w") as f:
        json.dump(protocol_number_data, f)
    print(f"Wrote: '{output_json_file}'")


def main(original_scorefile: Path, reproduce_scorefile: Path, num_protocols: int) -> None:
    """Save info about the lowest scRMSD decoy."""
    df1 = get_dataframe_from_pickle(original_scorefile)
    df2 = get_dataframe_from_pickle(reproduce_scorefile)
    protocol_number = num_protocols - 1 # Last protocol number, 0-indexed
    decoy_ids_1 = get_decoy_ids(df1, protocol_number)
    decoy_ids_2 = get_decoy_ids(df2, protocol_number)
    assert decoy_ids_1 == decoy_ids_2, f"Decoy IDs are not identical: {decoy_ids_1} != {decoy_ids_2}"

    protocol_number_data = {}
    for protocol_number in range(num_protocols):
        target_decoy_ids = decoy_ids_1[: (protocol_number + 1)]
        v1 = (
            df1
            .loc[df1["decoy_ids"].apply(lambda x: x == target_decoy_ids)]
            .pipe(assert_one_row) # Fail if >1 task was run
            .iloc[0]
        )
        v2 = (
            df2
            .loc[df2["decoy_ids"].apply(lambda x: x == target_decoy_ids)]
            .pipe(assert_one_row) # Fail if >1 task was run
            .iloc[0]
        )
        # Compute backbone heavy atom RMSD with Base64-encoded picked Pose
        # files with full atomic coordinate precision
        original_decoy = Path(v1["output_file"]).with_suffix(".b64_pose")
        reproduce_decoy = Path(v2["output_file"]).with_suffix(".b64_pose")
        bb_rmsd = get_bb_rmsd_nosuper(str(original_decoy), str(reproduce_decoy))
        # Compute sequence percent identity
        original_sequence = v1["sequence"]
        reproduce_sequence = v2["sequence"]
        sequence_percent_identity = get_sequence_percent_identity(original_sequence, reproduce_sequence)
        # Compute delta total_score
        original_total_score = v1["total_score"]
        reproduce_total_score = v2["total_score"]
        delta_total_score = float(reproduce_total_score - original_total_score)
        # Cache scores
        protocol_number_data[protocol_number] = {
            "bb_rmsd": bb_rmsd,
            "sequence_percent_identity": sequence_percent_identity,
            "delta_total_score": delta_total_score,
        }
    # Print data
    print_data(protocol_number_data)
    # Save data
    output_json_file = reproduce_scorefile.parent / "original_vs_reproduce_intermediate_decoy_comparison.json"
    write_data(output_json_file, protocol_number_data)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Analyze PyRosettaCluster results.",
    )
    parser.add_argument(
        "--original_scorefile",
        type=Path,
        required=True,
        help="The original PyRosettaCluster simulation output pickled `pandas.DataFrame` scorefile (i.e., a 'scores.bz2' file).",
    )
    parser.add_argument(
        "--reproduce_scorefile",
        type=Path,
        required=True,
        help="The reproduced PyRosettaCluster simulation output pickled `pandas.DataFrame` scorefile (i.e., a 'scores.bz2' file).",
    )
    parser.add_argument(
        "--num_protocols",
        type=int,
        required=False,
        default=6,
        help="The total number of user-defined PyRosetta protocols.",
    )
    args = parser.parse_args()
    main(
        args.original_scorefile,
        args.reproduce_scorefile,
        args.num_protocols,
    )

In [ ]:
# @title Compare the intermediate decoys from each user-defined PyRosetta protocol that were used to produce the original decoy of interest and the reproduced decoy of interest
if not os.getenv("DEBUG"):
    original_scorefile = original_output_path / "scores.bz2"
    reproduce_scorefile = reproduce_output_path / "scores.bz2"
    !cd {cloned_git_repo} && \
        pixi run --manifest-path {env_dir_manifest_path} \
        python -m compare \
        --original_scorefile {original_scorefile} \
        --reproduce_scorefile {reproduce_scorefile}

In [ ]:
# @title Maybe shutdown Google Colab (Optional)
if not os.getenv("DEBUG"):
    try:
        from google.colab import runtime
        runtime.unassign()
    except:
        pass


**Note: this notebook was adapted from the original code repository under MIT license:** https://github.com/klimaj/pyrosettacluster-examples
```
MIT License

Copyright (c) 2026 Jason C. Klima

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```

<!--NAVIGATION-->
< [PyRosettaCluster Tutorial 4. Ligand params](http://nbviewer.jupyter.org/github/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.10-PyRosettaCluster-Ligand-params.ipynb) | [Contents](toc.ipynb) | [Index](index.ipynb) | [Command Reference](http://nbviewer.jupyter.org/github/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/A.00-Appendix-A.ipynb) ><p><a href="https://colab.research.google.com/github/RosettaCommons/PyRosetta.notebooks/blob/master/notebooks/16.11-PyRosettaCluster-Foundry.ipynb"><img align="left" src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" title="Open in Google Colaboratory"></a>